# Data Wrangling
The purpose of this notebook is to prepare the dataset for analysis and modeling.

To do this, 4 things will be done in this notebook:
* A check for data integrity (missing values, correct data types)
* standardizing column names
* saving the reviewed dataset to the /data/clean/dataset.csv

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 999)

## Loading the Dataset

In [2]:
raw_df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
raw_df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## Checking for Missing Values

In [3]:
raw_df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

This dataset contains no missing values

## Data Types

In [4]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
for col in raw_df.columns:
    print(f"{col}\n{raw_df[col].unique()}\n")

customerID
<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str

gender
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

SeniorCitizen
[0 1]

Partner
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

tenure
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]

PhoneService
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtyp

* Drop customerID
* Changing column names to snake case
*  Churn, PaperlessBilling, PhoneService, Dependents can be integer bool
* TotalCharges should be float64
* Gender should be one hot encoded

## Removing customerID

In [6]:
raw_df1 = raw_df.drop("customerID", axis=1).copy()

## Changing Column Names to Snake Case

In [7]:
def to_snake_case(s):
    return ''.join(['_' + c.lower() if c.isupper() else c for c in s]).lstrip('_')

In [8]:
raw_df1.columns = [to_snake_case(c) for c in raw_df1.columns]

In [9]:
# Verifying Changes
raw_df1.columns

Index(['gender', 'senior_citizen', 'partner', 'dependents', 'tenure',
       'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_t_v', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'total_charges', 'churn'],
      dtype='str')

In [10]:
raw_df1 = raw_df1.rename(columns={"streaming_t_v": "streaming_tv"})

## Converting Bool Type Columns

In [11]:
bool_columns = ["churn", "phone_service", "dependents", "partner", "paperless_billing"]
raw_df2 = raw_df1.copy()

In [12]:
for col in bool_columns:
    raw_df2[col] = pd.Series(np.where(raw_df2[col].values == 'Yes', 1, 0),
          raw_df2.index)

## Checking Boolean Columns

In [13]:
raw_df2[bool_columns]

,churn,phone_service,dependents,partner,paperless_billing
0,0,0,0,1,1
1,0,1,0,0,0
2,1,1,0,0,1
3,0,0,0,0,0
4,1,1,0,0,1
...,...,...,...,...,...
7038,0,1,1,1,1
7039,0,1,1,1,1
7040,0,0,1,1,1
7041,1,1,0,1,1


## Converting total_charges data type to float64

In [14]:
raw_df3 = raw_df2.copy()

raw_df3["total_charges"] = raw_df3["total_charges"].astype('float64')

ValueError: could not convert string to float: ' '

just discovered observations where total charges is not listed. Since there are only 11 values missing, these rows can be dropped

In [15]:
raw_df4 = raw_df3.drop(
    raw_df3[raw_df3.total_charges == ' '].index
)

In [16]:
raw_df4['total_charges'] = raw_df4['total_charges'].astype('float')

In [17]:
raw_df4.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             7032 non-null   str    
 1   senior_citizen     7032 non-null   int64  
 2   partner            7032 non-null   int64  
 3   dependents         7032 non-null   int64  
 4   tenure             7032 non-null   int64  
 5   phone_service      7032 non-null   int64  
 6   multiple_lines     7032 non-null   str    
 7   internet_service   7032 non-null   str    
 8   online_security    7032 non-null   str    
 9   online_backup      7032 non-null   str    
 10  device_protection  7032 non-null   str    
 11  tech_support       7032 non-null   str    
 12  streaming_tv       7032 non-null   str    
 13  streaming_movies   7032 non-null   str    
 14  contract           7032 non-null   str    
 15  paperless_billing  7032 non-null   int64  
 16  payment_method     7032 non-null   str  

In [18]:
raw_df4.head(5)

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,Female,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,Male,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,Male,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,Male,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [19]:
raw_df4.to_csv("../data/clean/dataset.csv")